This file trains a simple encode / decoder architecture using PyTorch and exports it to ONNX format.
* First, the pytorch module is created. It consists of an encoder and a decoder, both implemented as simple feedforward neural networks.
Each of them simply has 3 strided convolutional layers (halfing the resolution in the encoder and increasing it in the decoder) followed by ReLU activations.
* Then, it is trained on a dataset of unlabelled images simply by reconstructing the input images from the latent representations.
* Finally, the trained model is exported to ONNX format for inference.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        return x

class Decoder(nn.Module):
    def __init__(self, out_channels=3):
        super().__init__()
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(32, out_channels, kernel_size=4, stride=2, padding=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.deconv1(x))
        x = self.relu(self.deconv2(x))
        x = self.relu(self.deconv3(x))
        return x

In [9]:
from datasets import load_dataset

dataset = load_dataset("bitmind/caltech-101")

import torchvision.transforms as T

# Prepare transforms and dataloader
transform = T.Compose([
    T.ToTensor(),
    T.Resize((128, 128)),
])

def preprocess1(example):
    img = example["image"]
    return {"image": transform(img)}

train_ds = dataset["train"].map(preprocess1)

Map: 100%|██████████| 9144/9144 [01:44<00:00, 87.50 examples/s] 


In [27]:
from torch.utils.data import DataLoader

from tqdm import tqdm

train_ds.set_format(type="torch", columns=["image"])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# Instantiate models
encoder = Encoder()
decoder = Decoder()
autoencoder = nn.Sequential(encoder, decoder)
autoencoder.train()

optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop for one epoch
for batch in tqdm(train_loader):
    imgs = batch["image"]
    optimizer.zero_grad()
    recon = autoencoder(imgs)
    loss = loss_fn(recon, imgs)
    loss.backward()
    optimizer.step()


100%|██████████| 572/572 [01:44<00:00,  5.46it/s]


In [ ]:
from matplotlib import pyplot as plt
from PIL import Image

# Load example image
img_path = "../../data/lantern.jpg"

image = Image.open(img_path)

plt.imshow(image)
plt.axis("off")
plt.show()


In [ ]:
image_tensor = transform(image).unsqueeze(0)
image_tensor.shape

with torch.no_grad():
    autoencoder.eval()
    encoder, decoder = autoencoder[0], autoencoder[1]

    latent = encoder(image_tensor)
    reconstructed = decoder(latent)
    
    reconstructed_img = reconstructed.squeeze(0).permute(1, 2, 0).numpy()

    plt.imshow(reconstructed_img)
    plt.axis("off")
    plt.show()


In [26]:
# Export the encoder
torch.onnx.export(encoder,
                  image_tensor,
                  "../../data/onnx/simple_encoder.onnx",
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'},
                                'output': {0: 'batch_size'}})

# Export the decoder
torch.onnx.export(decoder,
                  latent,
                  "../../data/onnx/simple_decoder.onnx",
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'},
                                'output': {0: 'batch_size'}})